# Финальные данные и модели

В предыдущих ноутбуках были рассмотрены модели, основанные на разных технологиях: линейная регрессия, деревья, нейронные сети, коллаборативные модели. 

В этом эксперименте необходимо выбрать лучшие модели из лучших, загрузить и проанализировать более тяжелый датасет под обучение модели.

Содержание:
1. Выбор лучшей модели
2. Загрузка и анализ данных для лучшей модели
3. Артефакты
4. Выводы

## 1. Выбор лучшей модели

В экспериментах 2-4 были рассмотрены модели, которые могут работать с cold-start. Они способны дать хоть какую-то персонализацию по немногочисленным пользовательским данным. В эксперименте 5 были рассмотрены коллаборативные модели: они дают мощные предсказания, если имеют историю оценок пользователя. Поэтому сравнение данных моделей не совсем честно по отношению к коллаборативным, так как в условиях cold-start они не могут выделиться.

Так как все модели показали примерно одинаковые результаты, то наравне с ошибкой будут рассматриваться и иные критерии: скорость обучения, скорость предсказания, размер модели и применимость к cold-start/warm-start.

### 1.1. Выбор cold-start модели

In [15]:
import os
import pandas as pd
import numpy as np

artifacts_path = os.path.join('..', 'artifacts')
exp02_results_df = pd.read_csv(os.path.join(artifacts_path, 'exp02', 'runs.csv'))
exp03_results_df = pd.read_csv(os.path.join(artifacts_path, 'exp03', 'runs.csv'))
exp04_results_df = pd.read_csv(os.path.join(artifacts_path, 'exp04', 'runs.csv'))

all_models_df = pd.concat(
    [
        exp02_results_df,
        exp03_results_df,
        exp04_results_df,
    ],
    ignore_index=True,
)

all_models_df = all_models_df[["name", "test_rmse", "test_mae"]]
all_models_df = all_models_df.sort_values("test_rmse").reset_index(drop=True)

all_models_df

,name,test_rmse,test_mae
0,CatBoostRegressor,1.010384,0.793933
1,NeuralCF,1.010899,0.799535
2,GradientBoostingRegressor,1.012868,0.802669
3,RandomForestRegressor,1.013138,0.799841
4,MLP,1.014628,0.803652
5,LassoGrid,1.015033,0.795284
6,ElasticNetGrid,1.015368,0.795079
7,RidgeGrid,1.019540,0.794322
8,RidgeRow,1.019555,0.794302
9,LinearRegression,1.019556,0.794302


#### Cold-start model - CatBoostRegressor
Эта модель показала лучший результат по метрикам, быстро предсказывает и обучается, интерпретируема.

### 1.2. Выбор warm-start модели

In [16]:
exp05_results_df = pd.read_csv(os.path.join(artifacts_path, 'exp05', 'runs.csv'))

exp05_results_df = exp05_results_df[["name", "test_rmse", "test_mae"]]
exp05_results_df = exp05_results_df.sort_values("test_rmse").reset_index(drop=True)

exp05_results_df

,name,test_rmse,test_mae
0,BiasBaseline,1.025338,0.798456
1,SVD,1.026400,0.797007
2,SVDpp,1.029354,0.799815
3,UserBased_KNNBaseline,1.033782,0.802892
4,ItemBased_KNNBaseline,1.033790,0.802798
5,NMF,1.035802,0.813517
6,UserBased_KNNWithMeans,1.078468,0.854010
7,ItemBased_KNNWithMeans,1.078972,0.854358


#### Warm-start model - SVD
Эта модель дала хороший результат по метрикам, способна строить сложные зависимости, быстра и интерпретируема

## 2. Загрузка и анализ данных для лучшей модели

Финальный датасет - ml-32m. Этот датасет идентичен по структуре ml-latest-small, содержит данные с 1995 по 2023 года. В нем собраны 32млн оценок на 87585 фильмов. Этот датасет поможет дать свежие рекомендации пользователям

Для обучения CatBoost будет использован ml-latest-small, так как модель не потянет 32 млн записей с 40 признаками, это гигабайты данных, которые должны поместиться в оперативной памяти.

Для обучения SVD будет использован ml-32m, так как ему не нужны никакие признаки

Для составления каталога также будет использован ml-32m

F. Maxwell Harper and Joseph A. Konstan. 2015. The MovieLens Datasets: History and Context. ACM Transactions on Interactive Intelligent Systems (TiiS) 5, 4: 19:1–19:19. https://doi.org/10.1145/2827872

### 2.1. Импорты

In [17]:
import requests #  библиотека для скачивания файлов по ссылке
import zipfile #  библиотека для работы с zip-архивами
import io #  библиотека для работы с потоками байтов
import gc

### 2.2. Скачивание архива и извлечение

In [ ]:
def download_and_extract(url: str, directory_to_extract_path: str, files_to_extract: list = None) -> None:
    """
    Скачивает архив и распаковывает только указанные файлы
    Параметры:
        url - ссылка для скачивания архива
        directory_to_extract_path - папка, куда распаковать
        files_to_extract - список имён файлов, которые нужны (если None - распаковывает всё)
    Возвращает:
      ничего
    """
    
    os.makedirs(directory_to_extract_path, exist_ok=True)

    response = requests.get(url)
    response.raise_for_status()

    with zipfile.ZipFile(io.BytesIO(response.content)) as zip_ref:
        if files_to_extract is None:
            zip_ref.extractall(directory_to_extract_path) #  распаковываем всё
        else:
            all_files = zip_ref.namelist() #  распаковываем только выбранные файлы

            for file in all_files:
                if os.path.basename(file) in files_to_extract:
                    zip_ref.extract(file, directory_to_extract_path)

    print(f'Файлы {files_to_extract} скачаны и распакованы в {directory_to_extract_path}')

url = 'https://files.grouplens.org/datasets/movielens/ml-32m.zip' #  ссылка для скачивания архива
directory_to_extract_path = os.path.join('..','data') #  путь для распаковки
files_to_extract = ['ratings.csv', 'movies.csv'] #  файлы для извлечения

download_and_extract(url, directory_to_extract_path, files_to_extract)

### 2.3. Выгрузка в датафреймы и проверка качества

In [ ]:
ds_folder_name = 'ml-32m' #  название папки датасета
directory_to_load_path = os.path.join(directory_to_extract_path, ds_folder_name) #  папка для загрузки

dfs = {} # словарь имя - датафрейм
for name in files_to_extract: # выгрузка в словарь
    file_path = os.path.join(directory_to_load_path, name)
    if os.path.exists(file_path):
        dfs[name] = pd.read_csv(file_path, sep=',')

def check_data_quality(df: pd.DataFrame) -> None:
  """
  Функция проверки качества данных датасета. Выводит информацию, помогающую понять, есть ли в датасете пропуски, дубликаты и др. несоответсвия
  Параметры:
    df - проверяемый датафрейм
  Возвращает:
    ничего
  """
  data_metrics = {
      'структура': lambda df: df.head(1),
      'размерность': lambda df: df.shape,
      'информация по столбцам': lambda df: df.info(),
      'доля пропусков': lambda df: df.isnull().sum() / len(df) * 100,
      'наличие дубликатов': lambda df: df.duplicated().sum(),
      'отрицательные значения': lambda df: (df.select_dtypes(include=['number']) < 0).sum()
  } # метрики

  for metric, function in data_metrics.items(): #  прогон по всем метрикам
      print('-'*20 + metric + '-'*20)
      print(function(df))

for name, df in dfs.items():
    print('\n' + '='*20 + name + '='*20 + '\n') #  вывод имени датасета
    check_data_quality(df)

### 2.4. Сборка необходимых датасетов

### 2.4.1. Вспомогательные функции

In [ ]:
def train_test_split_by_time(df: pd.DataFrame, split_column: str, test_size: float) -> tuple[pd.DataFrame, pd.DataFrame]:
  """
  Функция разделения по времени. Сортирует весь датасет по split_column.
  Делит записи в соотношении 1-test_size/test_size в train/test датафреймы соответственно.
  Параметры:
    df - датафрейм для деления
    split_column - колонка, по которой будет сортироваться датафрейм
    test_size - доля, которая пойдет в test
  Возвращает:
    train - выборка для обучения
    test - выборка для теста
  """
  #  Сортируем весь датасет по времени
  df_sorted = df.sort_values(split_column).reset_index(drop=True)

  #  Вычисляем индексы разделения
  train_end = int(len(df_sorted) * (1 - test_size))

  # Делим
  train = df_sorted.iloc[:train_end].copy()
  test = df_sorted.iloc[train_end:].copy()

  return train, test

### 2.4.2. Сырой user - movie - rating

In [ ]:
ratings_df = dfs["ratings.csv"].copy()
movies_df = dfs["movies.csv"].copy()

ratings_df["userId"] = ratings_df["userId"].astype("int32")
ratings_df["movieId"] = ratings_df["movieId"].astype("int32")
ratings_df["rating"] = ratings_df["rating"].astype("float32")
ratings_df["timestamp"] = ratings_df["timestamp"].astype("int32")

movies_df["movieId"] = movies_df["movieId"].astype("int32")

train, test = train_test_split_by_time(ratings_df, 'timestamp', 0.2) # train/val/test в пропорции 0.7/0.1/0.2
train, val = train_test_split_by_time(train, 'timestamp', 0.125)

train = train.drop(columns = ['timestamp'])
val = val.drop(columns = ['timestamp'])
test = test.drop(columns = ['timestamp'])

ml_32m_path = os.path.join('..', 'data', 'ml-32m')

train.to_csv(os.path.join(ml_32m_path, 'train.csv'), sep=',', index=False)
val.to_csv(os.path.join(ml_32m_path, 'val.csv'), sep=',', index=False)
test.to_csv(os.path.join(ml_32m_path, 'test.csv'), sep=',', index=False)

In [ ]:
print('\n' + '='*20 + 'train' + '='*20 + '\n')
check_data_quality(train)
print('\n' + '='*20 + 'val' + '='*20 + '\n')
check_data_quality(val)
print('\n' + '='*20 + 'test' + '='*20 + '\n')
check_data_quality(test)

### 2.4.3. Датасет с данными о фильмах

In [ ]:
# 1. Бинарные колонки жанров
genres_one_hot = (
    movies_df["genres"].str.get_dummies(sep="|")
    .drop(columns=["(no genres listed)"], errors="ignore")
    .astype("int8")
)
genre_list = genres_one_hot.columns.tolist()

movies_base_df = pd.concat([movies_df[["movieId", "title"]], genres_one_hot], axis=1)

# 2. Сумма и число оценок по фильму
movie_stats = (
    ratings_df.groupby("movieId")["rating"]
    .agg(rating_sum="sum", rating_count="count")
    .reset_index()
)
movies_base_df = movies_base_df.merge(movie_stats, on="movieId", how="left")
movies_base_df[["rating_sum", "rating_count"]] = movies_base_df[["rating_sum", "rating_count"]].fillna(0)

global_mean = ratings_df["rating"].mean()

# 3. Взвешенные жанровые средние
genres_mean = {}
for genre in genre_list:
    mask = movies_base_df[genre] == 1
    cnt = movies_base_df.loc[mask, "rating_count"].sum()
    genres_mean[genre] = (movies_base_df.loc[mask, "rating_sum"].sum() / cnt) if cnt > 0 else global_mean

# 4. Cреднее по его жанрам, иначе глобальное среднее
genre_matrix = movies_base_df[genre_list].to_numpy()
genre_vals = np.array([genres_mean[g] for g in genre_list])
gsum = genre_matrix @ genre_vals
gcnt = genre_matrix.sum(axis=1)
prior = np.where(gcnt > 0, gsum / gcnt, global_mean)

# 5. Регуляризованная средняя оценка
LAMBDA = 5.0
movies_base_df["movie_avg_rating"] = (
    (movies_base_df["rating_sum"].to_numpy() + LAMBDA * prior)
    / (movies_base_df["rating_count"].to_numpy() + LAMBDA)
)

# 6. Чистим временные колонки и пишем справочник
movies_base_df = movies_base_df[["movieId", "title", "movie_avg_rating"] + genre_list]
movies_base_df.to_csv(os.path.join('..', 'data', "movies_catalog.csv"), index=False)

In [ ]:
check_data_quality(movies_base_df)

### 2.5. Удаляем ненужные файлы

In [ ]:
for file in files_to_extract:
  path = os.path.join(directory_to_load_path, file)
  os.remove(path)

### 3. Артефакты

In [ ]:
import json
import os

artifacts_path = os.path.join("..", "artifacts", "exp06")
os.makedirs(artifacts_path, exist_ok=True)

selected_models = {
    "cold_start_model": {
        "name": "CatBoostRegressor",
        "source_experiment": "exp03_DecisionTree",
        "training_dataset": "ml-latest-small enriched_version"
    },
    "warm_start_model": {
        "name": "SVD",
        "source_experiment": "exp05_CollaborativeModels",
        "training_dataset": "ml-32m raw_version"
    },
}

with open(os.path.join(artifacts_path, "selected_models.json"), "w", encoding="utf-8") as f:
    json.dump(selected_models, f, ensure_ascii=False, indent=4)

## 4. Выводы

Результаты эксперимента:
1. Выбраны 2 лучшие модели для cold- и warm-start
2. Сырой датасет с train/val/test разбиением
3. Справочник фильмов

В итоговом проекте будет использована CatBoostRegressor модель для работы с новыми пользователями: по анкете, где пользователь оценивает 18 жанров, будет определена средняя оценка пользователя. Далее эта строка будет объединена с справочником фильмов и этот датасет отправится в predict модели. Также эту функцию можно использовать, если пользователь захочет переключиться на что-то нетипичное.

В проекте будет реализована функция оценки фильма, которая будет локально сохраняться.

После n-го количества оценок пользователь сможет воспользоваться SVD-моделью, чтобы получить персонализированные рекомендации. После каждых m оценок модель можно переобучить и улучшить рекомендации.